In [3]:
import pandas as pd
from Bio import SeqIO, SearchIO

In [4]:
%cd ../analysis/7.AWR-others/

[Errno 2] No such file or directory: '../analysis/7.AWR-others/'
/var2/Works/junhyeong/RXLR_landscape/analysis/7.AWR-others


/home/junhyeong/miniconda3/envs/biopython-env/lib/python3.13/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


In [26]:
wy1_query, wy2_query = SearchIO.parse("../5.AWR/awr.final.results", "hmmer3-text")

In [34]:
def func(query, tgt):
    entries = {}
    
    for hit in query:
        entry = hit.id

        if hit.id not in tgt:
            continue

        if entry not in entries:
            entries[entry] = []
        
        for i, hsp in enumerate(hit):
                entries[entry].append((i+1, hsp.bitscore))
                

    return entries

# HIT ORDER

In [35]:
wy1_wy2_wy1 = set(record.id for record in SeqIO.parse("./wy1-wy2-wy1.fasta", "fasta"))

In [36]:
len(wy1_wy2_wy1)

39

In [37]:
wy1_wy2_wy1_entries_w1 = func(wy1_query, wy1_wy2_wy1)
wy1_wy2_wy1_entries_w2 = func(wy2_query, wy1_wy2_wy1)

In [43]:
with open("prism.wy1.hmm.txt", "w") as f:
    for key, value in wy1_wy2_wy1_entries_w1.items():

        f.write(key + "\t" + '\t'.join([str(content[1]) for content in value]) + '\n')

In [44]:
with open("prism.wy2.hmm.txt", "w") as f:
    for key, value in wy1_wy2_wy1_entries_w2.items():

        f.write(key + "\t" + '\t'.join([str(content[1]) for content in value]) + '\n')

# Position

In [48]:
import numpy as np

In [115]:
def position_per_hmm_scores(seq_db, query, max_len):
    arr = np.zeros((max_len, len(seq_db)))
    i = 0
    for hit in query:
        entry = hit.id

        if entry not in seq_db:
            continue
        
        hit_range = []
        
        for hsp in hit.hsps:
            if hsp.bitscore <= 0:
                continue

            hit_range.append((hsp.hit_start, hsp.hit_end, hsp.bitscore))

        pt = 0
        for pos, c in enumerate(seq_db[entry]):
            if c == '-':
                arr[pos, i] = -1
                continue

            for r in hit_range:
                if pt >= r[0] and pt < r[1]:
                    arr[pos, i] = r[2]
                    break
            pt += 1
        i += 1
            
    return arr
        

In [116]:
wy1_query, wy2_query = SearchIO.parse("../5.AWR/awr.final.results", "hmmer3-text")
wy_query = SearchIO.parse("../5.AWR/awr.wy.results", "hmmer3-text")
lwy_query = SearchIO.parse("../5.AWR/awr.lwy.results", "hmmer3-text")

In [135]:
def trim(arr):
    return np.sum(arr == -1, axis=1)

## WY1-WY2-WY1

In [49]:
!pwd

/var2/Works/junhyeong/RXLR_landscape/analysis/7.AWR-others


In [50]:
!conda run -n homology-env hmmsearch -o awr.wy.results /var2/Works/junhyeong/RXLR_landscape/analysis/3.Clustering/3-1.domain_prediction/wy/WY.hmm ./wy1-wy2-wy1.fasta
!conda run -n homology-env hmmsearch -o awr.lwy.results /var2/Works/junhyeong/RXLR_landscape/analysis/3.Clustering/3-1.domain_prediction/wy/LWY.hmm ./wy1-wy2-wy1.fasta

In [64]:
!conda run -n homology-env mafft --thread 72 --maxiterate 1000 --quiet --globalpair wy1-wy2-wy1.fasta > wy1-wy2-wy1.align.fasta

In [141]:
!conda run -n phylo-env trimal -in wy1-wy2-wy1.align.fasta -gt 0.3 > wy1-wy2-wy1.align.trim70.fasta

In [112]:
seq_db = {}
for record in SeqIO.parse("./wy1-wy2-wy1.align.fasta", "fasta"):
    seq_db[record.id] = str(record.seq)

In [113]:
len(seq_db)

39

In [114]:
max_len = max(len(seq) for seq in seq_db.values())

In [134]:
wy1 = position_per_hmm_scores(seq_db, wy1_query, max_len)
wy2 = position_per_hmm_scores(seq_db, wy2_query, max_len)
# wy = position_per_hmm_scores(seq_db, wy_query, max_len)
# lwy = position_per_hmm_scores(seq_db, lwy_query, max_len)

In [136]:
wy1_trim = np.clip(wy1[trim(wy1) < 28], 0, None)
wy2_trim = np.clip(wy2[trim(wy2) < 28], 0, None)

In [138]:
wy1_mean = np.mean(wy1_trim, axis = 1)
wy2_mean = np.mean(wy2_trim, axis = 1)
# wy = np.mean(wy, axis = 1)
# lwy = np.mean(lwy, axis = 1)

In [139]:
d = np.stack([wy1_mean, wy2_mean], axis = 1)

In [140]:
pd.DataFrame(d).to_csv("prism.position_per_score.txt", sep = "\t", header = False)

## WY1-WY2

In [142]:
!conda run -n homology-env mafft --thread 72 --maxiterate 1000 --quiet --globalpair wy1-wy2.fasta > wy1-wy2.align.fasta

In [169]:
seq_db = {}
for record in SeqIO.parse("./wy1-wy2.align.fasta", "fasta"):
    seq_db[record.id] = str(record.seq)

In [170]:
len(seq_db)

114

In [171]:
max_len = max(len(seq) for seq in seq_db.values())

In [172]:
max_len

2143

In [173]:
wy1 = position_per_hmm_scores(seq_db, wy1_query, max_len)
wy2 = position_per_hmm_scores(seq_db, wy2_query, max_len)
# wy = position_per_hmm_scores(seq_db, wy_query, max_len)
# lwy = position_per_hmm_scores(seq_db, lwy_query, max_len)

In [178]:
wy1_trim = np.clip(wy1[trim(wy1) < 50], 0, None)
wy2_trim = np.clip(wy2[trim(wy2) < 50], 0, None)

In [179]:
wy1_mean = np.mean(wy1_trim, axis = 1)
wy2_mean = np.mean(wy2_trim, axis = 1)
# wy = np.mean(wy, axis = 1)
# lwy = np.mean(lwy, axis = 1)

In [180]:
wy1_mean.shape

(242,)

In [181]:
wy2_mean.shape

(242,)

In [182]:
d = np.stack([wy1_mean, wy2_mean], axis = 1)

In [183]:
pd.DataFrame(d).to_csv("prism.wy1-wy2.position_per_score.txt", sep = "\t", header = False)

## wy1-wy2-wy1-wy2-wy1

In [184]:
!conda run -n homology-env mafft --thread 72 --maxiterate 1000 --quiet --globalpair wy1-wy2-wy1-wy2-wy1.fasta > wy1-wy2-wy1-wy2-wy1.align.fasta

In [185]:
seq_db = {}
for record in SeqIO.parse("./wy1-wy2-wy1-wy2-wy1.align.fasta", "fasta"):
    seq_db[record.id] = str(record.seq)

In [186]:
len(seq_db)

6

In [187]:
max_len = max(len(seq) for seq in seq_db.values())

In [188]:
max_len

369

In [189]:
wy1 = position_per_hmm_scores(seq_db, wy1_query, max_len)
wy2 = position_per_hmm_scores(seq_db, wy2_query, max_len)
# wy = position_per_hmm_scores(seq_db, wy_query, max_len)
# lwy = position_per_hmm_scores(seq_db, lwy_query, max_len)

In [190]:
wy1_trim = np.clip(wy1[trim(wy1) < 5], 0, None)
wy2_trim = np.clip(wy2[trim(wy2) < 5], 0, None)

In [191]:
wy1_mean = np.mean(wy1_trim, axis = 1)
wy2_mean = np.mean(wy2_trim, axis = 1)
# wy = np.mean(wy, axis = 1)
# lwy = np.mean(lwy, axis = 1)

In [192]:
wy1_mean.shape

(347,)

In [193]:
wy2_mean.shape

(347,)

In [194]:
d = np.stack([wy1_mean, wy2_mean], axis = 1)

In [195]:
pd.DataFrame(d).to_csv("prism.wy1-wy2-wy1-wy2-wy1.position_per_score.txt", sep = "\t", header = False)